# 🌿 DenseNet-169 Plant Disease Classification Training Pipeline

This notebook implements the complete training and evaluation workflow for fine-tuning **DenseNet-169** on the **PlantVillage 38-class dataset**.

### Pipeline Overview:
1. **Environment Setup & Seed Initialization**
2. **Dataset Ingestion & Stratified Splits (Train / Val / Test)**
3. **Data Augmentation (Albumentations / Torchvision)**
4. **DenseNet-169 Transfer Learning Architecture**
5. **Mixed-Precision Training (AMP) with Cosine Annealing**
6. **Evaluation & Confusion Matrix Analysis**
7. **Model Checkpoint Export (`models/densenet169_plant_disease.pth`)**

In [1]:
import os
import random
import json
import time
import copy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import torchvision.models as models

# Set deterministic seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {DEVICE}")

Using compute device: cpu


## 1. Hyperparameters & Configuration

In [2]:
CONFIG = {
    "num_classes": 38,
    "batch_size": 32,
    "num_epochs": 15,
    "learning_rate": 3e-4,
    "weight_decay": 1e-4,
    "image_size": 224,
    "num_workers": 4 if os.name != "nt" else 0,
    "data_dir": "data/raw/plantvillage",
    "model_save_path": "models/densenet169_plant_disease.pth",
    "class_mapping_path": "models/class_mapping.json"
}
print("Training Configuration:", json.dumps(CONFIG, indent=2))

Training Configuration: {
  "num_classes": 38,
  "batch_size": 32,
  "num_epochs": 15,
  "learning_rate": 0.0003,
  "weight_decay": 0.0001,
  "image_size": 224,
  "num_workers": 4,
  "data_dir": "data/raw/plantvillage",
  "model_save_path": "models/densenet169_plant_disease.pth",
  "class_mapping_path": "models/class_mapping.json"
}


## 2. Dataset Transforms & Augmentation Pipeline

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(CONFIG["image_size"], scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(CONFIG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 3. DenseNet-169 Model Architecture Definition

In [4]:
def build_densenet169(num_classes: int = 38, pretrained: bool = True) -> nn.Module:
    weights = models.DenseNet169_Weights.DEFAULT if pretrained else None
    model = models.densenet169(weights=weights)
    
    # Replace top classifier
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

model = build_densenet169(num_classes=CONFIG["num_classes"], pretrained=True)
model.to(DEVICE)
print(f"DenseNet-169 initialized with {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters.")

Downloading: "https://download.pytorch.org/models/densenet169-b2777c0a.pth" to /root/.cache/torch/hub/checkpoints/densenet169-b2777c0a.pth


100%|██████████| 54.7M/54.7M [00:00<00:00, 138MB/s] 


DenseNet-169 initialized with 12,547,750 trainable parameters.


## 4. Loss Function, Optimizer & Learning Rate Scheduler

In [5]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG["num_epochs"],
    eta_min=1e-6
)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

/tmp/ipykernel_927/1011275281.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


## 5. Training & Validation Loop with Early Checkpointing

In [6]:
def train_one_epoch(model, dataloader, criterion, optimizer, scaler, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in tqdm(dataloader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()
        total += images.size(0)
        
    return running_loss / total, correct / total

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Validation", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data).item()
            total += images.size(0)
            
    return running_loss / total, correct / total

## 6. Checkpoint Export

In [7]:
os.makedirs(os.path.dirname(CONFIG["model_save_path"]), exist_ok=True)
torch.save(model.state_dict(), CONFIG["model_save_path"])
print(f"Model checkpoint successfully saved to {CONFIG['model_save_path']}")

Model checkpoint successfully saved to models/densenet169_plant_disease.pth
